# GDXU Technical Indicators: Project vs TA-Lib Comparison

Compare MACD and RSI calculations between project methods and official TA-Lib.

In [1]:
import polars as pl
import talib
from pathlib import Path
import sys

sys.path.insert(0, str(Path('..').resolve()))
from utils.indicators import calculate_macd_series, calculate_rsi_series

In [2]:
# Load data
df = pl.read_csv(Path("../data/GDXU_5min.csv"), try_parse_dates=True).sort('timestamp')
close_prices = df['close'].to_list()
close_series = df['close']

print(f"Loaded {len(df):,} rows")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

Loaded 106,630 rows
Date range: 2022-01-03 10:30:00+00:00 to 2025-11-20 00:00:00+00:00


## Calculate with Project Methods

In [3]:
# Project methods
macd_proj, signal_proj, hist_proj = calculate_macd_series(close_prices, fast_period=12, slow_period=26, signal_period=9)
rsi_proj = calculate_rsi_series(close_prices, period=14)

df_project = df.with_columns([
    pl.Series('macd_proj', macd_proj),
    pl.Series('signal_proj', signal_proj),
    pl.Series('hist_proj', hist_proj),
    pl.Series('rsi_proj', rsi_proj)
])

print("Project methods:")
df_project.select(['timestamp', 'close', 'macd_proj', 'signal_proj', 'rsi_proj']).tail(10)

Project methods:


timestamp,close,macd_proj,signal_proj,rsi_proj
"datetime[μs, UTC]",f64,f64,f64,f64
2025-11-19 22:25:00 UTC,174.003,0.023363,0.090495,45.754246
2025-11-19 22:30:00 UTC,176.0,0.125511,0.097499,56.673845
2025-11-19 22:50:00 UTC,175.89,0.195337,0.117066,56.005089
2025-11-19 23:15:00 UTC,175.7749,0.238635,0.14138,55.270164
2025-11-19 23:20:00 UTC,175.69,0.263067,0.165717,54.699974
2025-11-19 23:30:00 UTC,175.0,0.224168,0.177407,50.169966
2025-11-19 23:45:00 UTC,175.3,0.215068,0.18494,52.030072
2025-11-19 23:50:00 UTC,175.7,0.237397,0.195431,54.470483
2025-11-19 23:55:00 UTC,177.0,0.355889,0.227523,61.352065


## Calculate with TA-Lib

In [4]:
# TA-Lib
macd_talib, signal_talib, hist_talib = talib.MACD(close_series, fastperiod=12, slowperiod=26, signalperiod=9)
rsi_talib = talib.RSI(close_series, timeperiod=14)

df_talib = df_project.with_columns([
    macd_talib.alias('macd_talib'),
    signal_talib.alias('signal_talib'),
    hist_talib.alias('hist_talib'),
    rsi_talib.alias('rsi_talib')
])

print("TA-Lib:")
df_talib.select(['timestamp', 'close', 'macd_talib', 'signal_talib', 'rsi_talib']).tail(10)

TA-Lib:


timestamp,close,macd_talib,signal_talib,rsi_talib
"datetime[μs, UTC]",f64,f64,f64,f64
2025-11-19 22:25:00 UTC,174.003,0.023363,0.090495,45.754246
2025-11-19 22:30:00 UTC,176.0,0.125511,0.097499,56.673845
2025-11-19 22:50:00 UTC,175.89,0.195337,0.117066,56.005089
2025-11-19 23:15:00 UTC,175.7749,0.238635,0.14138,55.270164
2025-11-19 23:20:00 UTC,175.69,0.263067,0.165717,54.699974
2025-11-19 23:30:00 UTC,175.0,0.224168,0.177407,50.169966
2025-11-19 23:45:00 UTC,175.3,0.215068,0.18494,52.030072
2025-11-19 23:50:00 UTC,175.7,0.237397,0.195431,54.470483
2025-11-19 23:55:00 UTC,177.0,0.355889,0.227523,61.352065


## Comparison: Project vs TA-Lib

In [5]:
# Calculate differences
df_comparison = df_talib.with_columns([
    (pl.col('macd_proj') - pl.col('macd_talib')).alias('macd_diff'),
    (pl.col('signal_proj') - pl.col('signal_talib')).alias('signal_diff'),
    (pl.col('hist_proj') - pl.col('hist_talib')).alias('hist_diff'),
    (pl.col('rsi_proj') - pl.col('rsi_talib')).alias('rsi_diff')
])

print("Side-by-side comparison (last 10 rows):")
df_comparison.select([
    'timestamp', 'close',
    'macd_proj', 'macd_talib', 'macd_diff',
    'rsi_proj', 'rsi_talib', 'rsi_diff'
]).tail(10)

Side-by-side comparison (last 10 rows):


timestamp,close,macd_proj,macd_talib,macd_diff,rsi_proj,rsi_talib,rsi_diff
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,f64
2025-11-19 22:25:00 UTC,174.003,0.023363,0.023363,0.0,45.754246,45.754246,0.0
2025-11-19 22:30:00 UTC,176.0,0.125511,0.125511,0.0,56.673845,56.673845,0.0
2025-11-19 22:50:00 UTC,175.89,0.195337,0.195337,0.0,56.005089,56.005089,0.0
2025-11-19 23:15:00 UTC,175.7749,0.238635,0.238635,0.0,55.270164,55.270164,-7.1054e-15
2025-11-19 23:20:00 UTC,175.69,0.263067,0.263067,0.0,54.699974,54.699974,7.1054e-15
2025-11-19 23:30:00 UTC,175.0,0.224168,0.224168,0.0,50.169966,50.169966,7.1054e-15
2025-11-19 23:45:00 UTC,175.3,0.215068,0.215068,0.0,52.030072,52.030072,0.0
2025-11-19 23:50:00 UTC,175.7,0.237397,0.237397,0.0,54.470483,54.470483,7.1054e-15
2025-11-19 23:55:00 UTC,177.0,0.355889,0.355889,0.0,61.352065,61.352065,7.1054e-15


In [6]:
# Summary statistics
print("\nDifference Statistics:")
df_comparison.select(['macd_diff', 'signal_diff', 'hist_diff', 'rsi_diff']).describe()


Difference Statistics:


statistic,macd_diff,signal_diff,hist_diff,rsi_diff
str,f64,f64,f64,f64
"""count""",106605.0,106597.0,106597.0,106616.0
"""null_count""",25.0,33.0,33.0,14.0
"""mean""",NaN,-0.000005,0.000003,9.7885e-18
"""std""",NaN,0.00048,0.000275,6.0919e-15
"""min""",-0.035838,-0.076595,0.0,-2.8422e-14
"""25%""",0.0,0.0,0.0,0.0
"""50%""",0.0,0.0,0.0,0.0
"""75%""",0.0,0.0,0.0,0.0
"""max""",0.0,0.0,0.040757,2.8422e-14


In [7]:
# Maximum absolute differences
print("\nMaximum Absolute Differences:")
df_comparison.select([
    pl.col('macd_diff').abs().max().alias('max_macd_diff'),
    pl.col('signal_diff').abs().max().alias('max_signal_diff'),
    pl.col('hist_diff').abs().max().alias('max_hist_diff'),
    pl.col('rsi_diff').abs().max().alias('max_rsi_diff')
])


Maximum Absolute Differences:


max_macd_diff,max_signal_diff,max_hist_diff,max_rsi_diff
f64,f64,f64,f64
0.035838,0.076595,0.040757,2.8422e-14


######################### Code to test using TALIB and dequeu ####################
